# Seguimiento de jugadores robusto usando número y equipo

Este notebook muestra la integración del sistema final de seguimiento de jugadores desarrollado en el TFM.

El sistema combina:
- BoT-SORT como tracker base
- Detección de dorsales
- OCR de dorsales
- Clasificación de equipos utilizando la zona del uniforme de los jugadores
- Uso de equipos y dorsales para mejorar el seguimiento de los jugadores

Los pesos de los modelos, los vídeos y los datasets no se incluyen en este repositorio. Las clases desarrolladas para el sistema se encuentran en las carpetas incluidas junto a este notebook.

Se cargan las clases de las subcarpetas y las funciones utilizadas

In [ ]:
import sys
from pathlib import Path

# Subcarpetas con clases que se utilizan en este notebook
rutas = ["MejoraSeguimiento", "Auxiliares", "VisionArtificial"]

# Se añaden las rutas al path para poder importar las clases
for ruta in rutas:
     ruta_completa = str(Path.cwd() / ruta)
     if ruta_completa not in sys.path:
         sys.path.append(ruta_completa)

# Se importan las clases necesarias para el notebook
from ManejadorJugadores import ManejadorJugadores
from Detector import Detector
from OCR import OCR
from Tracker import Tracker
from Equipo import ClasificadorEquipo
from bbox import calcular_area_de_bbox_en_otra, recortar_bbox

Se importan las librerías necesarias para ejecutar el notebook

In [ ]:
import torch
from tqdm.notebook import tqdm

# IoU para comparar bboxes
from torchvision.ops import box_iou

import cv2

Se crea una función para asociar los dorsales detectados en la imagen con los bounding boxes de los tracks presentes en la imagen.

In [ ]:
def asociar_dorsales_a_tracks(bboxes_dorsales: list[tuple[int, int, int, int]], tracks_actuales: list, tracker: Tracker, porcentaje_min=0.8) -> dict:
    """Asocia los dorsales detectados a los tracks actuales (usando sus bounding boxes). Devuelve un diccionario con el ID del track como clave y el bbox del dorsal asociado como valor (4 coordenadas)."""
    
    # Se guarda en pares:
    # - Índice del bbox del dorsal para la lista bboxes_dorsales
    # - ID del track
    # - Score de asociación (porcentaje de la bbox del dorsal dentro de la del track, distancia entre centros)
    pares = []
    for i, bbox_dorsal in enumerate(bboxes_dorsales):
        # Obtener el centro del bbox del dorsal
        x1_dorsal, y1_dorsal, x2_dorsal, y2_dorsal = tuple(map(int, bbox_dorsal[:4]))
        cx_dorsal = (x1_dorsal + x2_dorsal) / 2
        cy_dorsal = (y1_dorsal + y2_dorsal) / 2

        for track in tracks_actuales:
            # Obtener el ID del track
            track_id = tracker.obtener_id_track(track)
            # Obtener la parte del bbox del track que suele ser el centro del dorsal
            x1_track, y1_track, x2_track, y2_track = tuple(map(int, tracker.obtener_bbox_track(track)))
            cx_track = (x1_track + x2_track) / 2
            h_track = y2_track - y1_track
            cy_track = y1_track + h_track * 0.35  # Se asume que el dorsal suele estar en la parte superior del track

            # Calcular el porcentaje de área de dorsal en el bbox del track
            porcentaje = calcular_area_de_bbox_en_otra([x1_dorsal, y1_dorsal, x2_dorsal, y2_dorsal], [x1_track, y1_track, x2_track, y2_track])

            # Si sobresale mucho, no se asocia
            if porcentaje < porcentaje_min:
                continue
            
            # Se calcula la distancia entre los centros del dorsal y el track
            dist_x = (cx_dorsal - cx_track)
            dist_y = (cy_dorsal - cy_track)
            dist = dist_x**2 + dist_y**2

            # Si el dorsal está en la parte baja del track, no se asocia
            if cy_dorsal > (y1_track + y2_track) / 2:
                continue

            score = (porcentaje, dist)
            pares.append((i, track_id, score))
        
    # Asociar los que tienen mayor porcentaje
    pares.sort(key=lambda x: (-x[2][0], x[2][1]))
    asociaciones = {}
    track_ids_asociados = set()
    numeros_asociados = set()
    for i, track_id, score in pares:
        if track_id not in track_ids_asociados and i not in numeros_asociados:
            asociaciones[track_id] = tuple(map(int, bboxes_dorsales[i][:4]))
            track_ids_asociados.add(track_id)
            numeros_asociados.add(i)
    return asociaciones

Se indican algunos parámetros de los vídeos:
- Vídeo de salida del sistema
- Vídeo del partido de entrada al sistema
- Números de los jugadores del equipo A
- Color del uniforme (aproximado) del equipo A
- Números de los jugadores del equipo B
- Color del uniforme (aproximado) del equipo B

In [ ]:
VIDEO_OUTPUT = "./output.mp4"
VIDEO_SEGUIMIENTO_PATH_INPUT = "./input.mp4"
TEAM_A_NUMBERS = [23, 35, 27, 30, 0] 
TEAM_A_COLOR_BGR = (255, 255, 255) 
TEAM_B_NUMBERS = [7, 17, 21, 5, 11]
TEAM_B_COLOR_BGR = (0, 0, 0)

Se crea una función para procesar un vídeo de un partido

In [ ]:
# Frames iniciales que se pasan sin hacer nada
FRAMES_INICIALES = 0
# Frames que se usan para almacenar crops de jugadores y entrenar el clasificador de equipo
# (No se realiza seguimiento durante estos frames)
FRAMES_FIT_EQUIPO = 50

# Colores para dibujar resultados en el frame
color_fondo = (255, 255, 255) # Blanco para el fondo del texto
color_track = (0, 255, 0) # Verde para el bounding box (si no tiene equipo asociado)
color_equipoA = TEAM_A_COLOR_BGR # Color del equipo A para el bounding box de los jugadores del equipo A
color_equipoB = TEAM_B_COLOR_BGR # Color del equipo B para el bounding box de los jugadores del equipo B
color_texto = (0, 0, 0) # Negro para el texto

def procesar_video_equipos(video_path_input, video_path_output, detector, ocr, allowed_numbers_A = None, allowed_numbers_B = None, max_frames = None, equipo_A_BGR = None, equipo_B_BGR = None):
    """Procesa el vídeo `video_path_input` y guarda el resultado en `video_path_output`.
    Parámetros de entrada:
    - video_path_input: Ruta al vídeo de entrada
    - video_path_output: Ruta al vídeo de salida
    - detector: Detector de jugadores
    - ocr: OCR para reconocer los números de los dorsales
    - allowed_numbers_A: Lista de números permitidos para el equipo A
    - allowed_numbers_B: Lista de números permitidos para el equipo B
    - max_frames: Número máximo de frames a procesar. Si no se indica o se indica None, se procesa el vídeo de entrada en completo
    - equipo_A_BGR: Color del equipo A en formato BGR
    - equipo_B_BGR: Color del equipo B en formato BGR
    """
    # Se crea el manejador de jugadores
    manejador_jugadores = ManejadorJugadores(numeros_jugadores_A=allowed_numbers_A, numeros_jugadores_B=allowed_numbers_B)
    clasificador_equipo = None
    # Se crea el tracker
    tracker = Tracker()

    # Configurar el video de entrada y el de salida    
    cap = cv2.VideoCapture(video_path_input)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_path_output, fourcc, fps, (width, height))

    # Se calcula el número de frames a procesar para la barra de progreso
    if max_frames is not None:
        total_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), max_frames)
    else:
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Se crea la barra de progreso
    pbar = tqdm(total=total_frames, desc="Procesando vídeo")

    # Lista para almacenar crops para entrenar el clasificador de equipos
    crops_de_jugadores = []

    # Número de frames procesados
    frame_count = 0

    # Pasar FRAMES_INICIALES frames antes de empezar
    for _ in range(FRAMES_INICIALES):
        ret, frame = cap.read()
        if not ret:
            break

    # Procesar el vídeo de entrada
    while True:
        # Obtener el frame actual
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames is not None and frame_count >= max_frames:
            break
        frame_count += 1
        frame_copy = frame.copy()

        # Detectar jugadores y dorsales en el frame actual
        bboxes_jugadores, bboxes_resto = detector.detectar_jugadores_data(frame)
        bboxes_dorsales = [bbox for bbox in bboxes_resto if int(bbox[5]) == Detector.DORSAL_ID]

        # Actualizar tracker
        tracks_actuales = tracker.actualizar(frame)
        tracks_actuales = tracks_actuales[0]

        # Obtener los track_ids de los tracks vistos en el frame actual
        tracks_actuales_ids = [tracker.obtener_id_track(track) for track in tracks_actuales]

        # Almacenar crops de los jugadores para entrenar clasificador de equipos
        if frame_count <= FRAMES_FIT_EQUIPO:
            # Nos quedamos con los de más de 0.9 de confianza para entrenar el clasificador con crops de muy buena calidad
            bboxes_jugadores_conf =  bboxes_jugadores[bboxes_jugadores[:, 4] >= 0.9]
            # Si se detecta 1 jugador
            if len(bboxes_jugadores_conf) == 1:
                crop = recortar_bbox(frame, bboxes_jugadores_conf[0][:4])
                if crop is not None:
                    crops_de_jugadores.append(crop)
            # Si se detectan múltiples jugadores, no se añaden los que tienen mucho IoU con otros (podría confundir al kmeans)
            if len(bboxes_jugadores_conf) > 1:
                bboxes_jugadores_tensor = torch.tensor(bboxes_jugadores_conf)[:, :4]  # Nos quedamos sólo con las coordenadas de los bboxes
                
                # Por cada bbox, se decide si se guarda o no
                iou_matriz = box_iou(bboxes_jugadores_tensor, bboxes_jugadores_tensor)
                for i, bbox in enumerate(bboxes_jugadores_conf):
                    iou_max = torch.max(iou_matriz[i][torch.arange(len(bboxes_jugadores_conf)) != i])
                    if iou_max < 0.4:
                        crop = recortar_bbox(frame, bbox[:4])
                        if crop is not None:
                            crops_de_jugadores.append(crop)

            # Entrenar si estamos en el último frame de almacenar los crops
            if frame_count == FRAMES_FIT_EQUIPO:
                clasificador_equipo = ClasificadorEquipo(crops_jugadores=crops_de_jugadores, equipo_A_BGR=equipo_A_BGR, equipo_B_BGR=equipo_B_BGR)

        # Asociar dorsales a tracks
        asociaciones = asociar_dorsales_a_tracks(bboxes_dorsales, tracks_actuales, tracker)

        # Usar asociaciones para leer los dorsales y hacer dos listas: tracks_ids y números asociados (None si no se ha podido asociar número)
        tracks_ids = []
        numeros_asociados = []
        for track in tracks_actuales:
            track_id = tracker.obtener_id_track(track)
            tracks_ids.append(track_id)

            if track_id in asociaciones:
                bbox_dorsal = asociaciones[track_id]
                recorte = recortar_bbox(frame, bbox_dorsal)
                if recorte is not None:
                    numero = ocr.obtener_numero_dorsal(recorte)
                    numeros_asociados.append(numero)
                else:
                    numeros_asociados.append(None)
            else:
                numeros_asociados.append(None)

        # Obtener el equipo de cada jugador usando el clasificador de equipo
        bounding_boxes_manejador = [tracker.obtener_bbox_track(track) for track in tracks_actuales]
        equipos_manejador = []
        for bbox in bounding_boxes_manejador:
            x1, y1, x2, y2 = map(int, bbox)
            crop = recortar_bbox(frame, [x1, y1, x2, y2])
            if crop is not None and clasificador_equipo is not None:
                equipo = clasificador_equipo.clasificar(crop)
                equipos_manejador.append(equipo)
            else:
                equipos_manejador.append(None)

        # Actualizar manejador de jugadores
        jugadores_actual, tracks_bien_ordenados = manejador_jugadores.actualizar_con_equipo(frame_actual=frame_count, track_ids=tracks_ids, numeros_detectados=numeros_asociados, bounding_boxes=bounding_boxes_manejador, equipos_detectados=equipos_manejador)

        # Asociar tracks del frame actual con el jugador y el equipo correspondiente
        # Lista (track, jugador) para cada track_id en tracks_actuales_ids y jugador en jugadores_actual que tenga track_id asociado igual a track_id
        asociaciones_track_jugador = []
        for track in tracks_actuales:
            track_id = tracker.obtener_id_track(track)
            if track_id in tracks_actuales_ids:
                for jugador in jugadores_actual:
                    if jugador.track_id == track_id:
                        equipo = jugador.equipo
                        asociaciones_track_jugador.append((track, jugador, equipo))
                        break

        # Se pinta el bbox, el número y el equipo de cada asociaciones_track_jugador
        for track, jugador, equipo in asociaciones_track_jugador:
            x1, y1, x2, y2 = map(int, tracker.obtener_bbox_track(track))
            color_bbox = color_equipoA if equipo == "A" else color_equipoB if equipo == "B" else color_track
            cv2.rectangle(frame_copy, (x1, y1), (x2, y2), color_bbox, 2)
            equipo_texto = f'|{equipo}' if equipo is not None else ''
            cv2.rectangle(frame_copy, (x1, y1 - 30), (x2, y1), color_fondo, -1)
            numero_texto = f'#{jugador.numero}' if jugador.numero is not None else ''
            cv2.putText(frame_copy, f'{numero_texto}{equipo_texto}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_texto, 2)

        # Pintar también los colores de los equipos de los tracks vistos en el frame
        for bbox in bboxes_jugadores:
            x1, y1, x2, y2 = map(int, bbox[:4])
            crop = recortar_bbox(frame, [x1, y1, x2, y2])
            if crop is not None and clasificador_equipo is not None:
                equipo = clasificador_equipo.clasificar(crop)
                color_bbox = color_equipoA if equipo == "A" else color_equipoB if equipo == "B" else color_track
            else:
                color_bbox = color_track
            cv2.rectangle(frame_copy, (x1, y1), (x2, y2), color_bbox, 1)    
        
        # Pintar también los dorsales detectados sin asociar
        for bbox_dorsal in bboxes_dorsales:
            x1, y1, x2, y2 = map(int, bbox_dorsal[:4])
            cv2.rectangle(frame_copy, (x1, y1), (x2, y2), (0, 0, 255), 1) # Rojo para los dorsales sin asociar
            
            texto_dorsal = ocr.obtener_numero_dorsal(recortar_bbox(frame, [x1, y1, x2, y2]))
            cv2.rectangle(frame_copy, (x1, y1 - 20), (x2, y1), color_fondo, -1)
            cv2.putText(frame_copy, f'{texto_dorsal}', (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_texto, 1)

        # Escribir el frame procesado
        out.write(frame_copy)
        pbar.update(1)
    
    # Liberar recursos
    cap.release()
    out.release()

Se usa CUDA si está disponible la GPU (para acelerar el proceso). Si no, se usa la CPU.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

Se ejecuta el programa

In [ ]:
procesar_video_equipos(
    video_path_input=VIDEO_SEGUIMIENTO_PATH_INPUT, 
    video_path_output=VIDEO_OUTPUT, 
    detector=Detector(device=device),
    ocr=OCR(device=device),
    allowed_numbers_A=TEAM_A_NUMBERS,
    allowed_numbers_B=TEAM_B_NUMBERS,
    max_frames=500,
    equipo_A_BGR=TEAM_A_COLOR_BGR,
    equipo_B_BGR=TEAM_B_COLOR_BGR
    )
